# Notebook 8. TurboQuant, a study notebook

The seven notebooks before this one quantize weights with an affine grid: a scale and a zero-point per group of values. Keys and values in a KV cache are quantized vector by vector as tokens arrive, and at two or three bits per coordinate an affine grid per vector loses a lot. TurboQuant (Zandieh, Daliri, Hadian and Mirrokni, ICLR 2026, [arXiv:2504.19874](https://arxiv.org/abs/2504.19874)) takes a different route in three steps: rotate the vector by a random orthogonal matrix so that every coordinate follows one known distribution, quantize each coordinate with the scalar quantizer that is optimal for that distribution, and correct the remaining error with one sign bit per coordinate so that inner products with the quantized vector are unbiased. This notebook implements the three steps on small vectors and checks the numbers the paper states for each.

Prerequisites are Notebooks 1 and 6. Nothing here needs a model or a download.

In [ ]:
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. The setting

A vector $x \in \mathbb{R}^d$ with $\lVert x \rVert_2 = 1$ is to be stored with $b$ bits per coordinate. The paper measures a quantizer by its distortion, the expected squared reconstruction error $D = \mathbb{E}\,\lVert x - \tilde{x} \rVert_2^2$, and states a lower bound of $1/4^b$ for any quantizer of unit vectors. For attention, the quantity that matters is the inner product $\langle y, x \rangle$ between a query $y$ and a stored key $x$, and the paper's second variant targets the error of that inner product instead.

The baseline is what Notebooks 1 and 2 would do with a vector: an asymmetric $b$-bit grid between the vector's minimum and maximum, one scale per vector.

In [ ]:
d = 128


def unit(X):
    return X / X.norm(dim=1, keepdim=True)


def affine_per_vector(X, bits):
    q_max = 2**bits - 1
    lo, hi = X.amin(dim=1, keepdim=True), X.amax(dim=1, keepdim=True)
    s = (hi - lo) / q_max
    return lo + torch.round((X - lo) / s).clamp(0, q_max) * s


def distortion(X, X_hat):
    return (X - X_hat).pow(2).sum(dim=1).mean().item()


torch.manual_seed(0)
X_sphere = unit(torch.randn(5000, d))                 # vectors uniform on the sphere
X_kv = torch.randn(5000, d) * 0.3                    # vectors with four large coordinates,
X_kv[:, :4] *= 20                                    # the shape of keys with outlier channels
X_kv = unit(X_kv)

print("affine per-vector distortion (lower bound 1/4^b in parentheses)")
for bits in range(1, 5):
    d_sphere = distortion(X_sphere, affine_per_vector(X_sphere, bits))
    d_kv = distortion(X_kv, affine_per_vector(X_kv, bits))
    print(f"  b={bits}: sphere {d_sphere:.4f}, outlier vectors {d_kv:.4f}   ({1 / 4**bits:.4f})")

Four large coordinates set the scale for all 128, and the affine grid's distortion on those vectors is several times its distortion on featureless ones. TurboQuant's first step removes that dependence on the shape of the input.

## 2. Rotation

Multiplying $x$ by a random orthogonal matrix $\Pi$ preserves norms and inner products and makes $\Pi x$ uniformly distributed on the sphere whatever $x$ was. Lemma 1 of the paper gives the density of one coordinate of a uniform unit vector,

$$f_X(t) = \frac{\Gamma(d/2)}{\sqrt{\pi}\,\Gamma\!\left((d-1)/2\right)}\,(1 - t^2)^{(d-3)/2}, \qquad t \in [-1, 1],$$

which is close to $\mathcal{N}(0, 1/d)$ for large $d$. The paper generates $\Pi$ by QR decomposition of a matrix with i.i.d. normal entries. The cell rotates one vector with four outlier coordinates by 400 different $\Pi$ and compares the histogram of the resulting coordinates with $f_X$.

In [ ]:
import math


def random_rotation(d, generator=None):
    Q, _ = torch.linalg.qr(torch.randn(d, d, generator=generator))
    return Q


def coordinate_density(t, d):
    log_c = (torch.lgamma(torch.tensor(d / 2)) - 0.5 * math.log(math.pi)
             - torch.lgamma(torch.tensor((d - 1) / 2)))
    return torch.exp(log_c) * (1 - t**2).clamp(min=0) ** ((d - 3) / 2)


gen = torch.Generator().manual_seed(1)
x = X_kv[0]
rotated = torch.stack([random_rotation(d, gen) @ x for _ in range(400)])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(range(d), x.numpy(), width=1)
axes[0].set_title("x: four coordinates carry most of the norm")
axes[0].set_xlabel("coordinate")
ts = torch.linspace(-0.4, 0.4, 400)
axes[1].hist(rotated.flatten().numpy(), bins=80, density=True, alpha=0.6, label="coordinates of Πx")
axes[1].plot(ts, coordinate_density(ts, d), "r-", label="f_X from Lemma 1")
axes[1].plot(ts, torch.exp(-ts**2 * d / 2) * math.sqrt(d / (2 * math.pi)), "k--", label="N(0, 1/d)")
axes[1].set_title("after rotation")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"largest |coordinate| before rotation {x.abs().max():.3f}, after rotation (median over Π) "
      f"{rotated.abs().amax(dim=1).median():.3f}")

After rotation the coordinates follow $f_X$ regardless of where the norm of $x$ was concentrated. One scalar quantizer designed for $f_X$ therefore serves every vector, and no per-vector scale is needed.

## 3. The optimal scalar quantizer

For a known density, the $2^b$ levels that minimize the expected squared error are found by the Lloyd-Max iteration, which is $k$-means in one dimension: assign every value to its nearest level, move each level to the mean of the values assigned to it, repeat. The paper solves this numerically for $f_X$ once per bit-width and stores the levels (its equation (4)). The cell does the same on a fine grid, then checks three statements from the paper: the one-bit levels are $\pm\sqrt{2/\pi}/\sqrt{d}$, the two-bit levels are $\pm 0.453/\sqrt{d}$ and $\pm 1.51/\sqrt{d}$, and the distortion $D_{\mathrm{mse}} = d \cdot C(f_X, b)$ is about 0.36, 0.117, 0.03 and 0.009 for $b = 1$ to $4$, against the bound $(\sqrt{3}\pi/2)/4^b$ of Theorem 1 and the lower bound $1/4^b$.

In [ ]:
grid_t = torch.linspace(-1, 1, 40001)
grid_w = coordinate_density(grid_t, d)
grid_w = grid_w / grid_w.sum()                          # probability mass per grid point


def lloyd_max(t, w, n_levels, n_iter=200):
    cdf = torch.cumsum(w, 0)
    starts = [(cdf >= (k + 0.5) / n_levels).nonzero()[0, 0] for k in range(n_levels)]
    levels = torch.stack([t[i] for i in starts])          # start from the quantiles
    for _ in range(n_iter):
        nearest = (t[:, None] - levels[None, :]).abs().argmin(dim=1)
        for k in range(n_levels):
            mask = nearest == k
            if w[mask].sum() > 0:
                levels[k] = (w[mask] * t[mask]).sum() / w[mask].sum()
    nearest = (t[:, None] - levels[None, :]).abs().argmin(dim=1)
    cost = (w * (t - levels[nearest]) ** 2).sum().item()
    return levels, cost


codebooks = {}
print("b   D_mse = d*C   paper   upper bound   lower bound   levels * sqrt(d)")
for bits in range(1, 5):
    levels, cost = lloyd_max(grid_t, grid_w, 2**bits)
    codebooks[bits] = levels
    shown = [round(v, 3) for v in (levels * math.sqrt(d)).tolist()]
    shown = shown if bits < 3 else shown[:4] + ["..."]
    upper = math.sqrt(3) * math.pi / 2 / 4**bits
    print(f"{bits}   {d * cost:10.4f}   {[0.36, 0.117, 0.03, 0.009][bits - 1]:<6}  "
          f"{upper:11.4f}   {1 / 4**bits:11.4f}   {shown}")
one_bit = math.sqrt(2 / math.pi)
print(f"\none-bit level from the paper: ±sqrt(2/pi) = ±{one_bit:.3f}, times 1/sqrt(d)")

The computed distortions and levels agree with the paper to the precision it gives. With the levels fixed, the quantizer is a lookup.

$$\mathrm{Quant}_{\mathrm{mse}}(x) = \arg\min_k \lvert (\Pi x)_j - c_k \rvert \text{ for each } j, \qquad \mathrm{DeQuant}_{\mathrm{mse}}(\mathrm{idx}) = \Pi^T c_{\mathrm{idx}}.$$

The cell runs it on the two sets of vectors from Section 1 next to the affine baseline.

In [ ]:
Pi = random_rotation(d, gen)


def tq_mse_quantize(X, Pi, levels):
    Y = X @ Pi.T                                         # rotate
    return (Y[..., None] - levels).abs().argmin(dim=-1)  # nearest level per coordinate


def tq_mse_dequantize(idx, Pi, levels):
    return levels[idx] @ Pi                              # look up, rotate back


print("distortion, b   TurboQuant_mse (sphere, outlier vectors)   affine (sphere, outlier vectors)")
for bits in range(1, 5):
    levels = codebooks[bits]
    tq = [distortion(X, tq_mse_dequantize(tq_mse_quantize(X, Pi, levels), Pi, levels))
          for X in (X_sphere, X_kv)]
    af = [distortion(X, affine_per_vector(X, bits)) for X in (X_sphere, X_kv)]
    print(f"            {bits}   {tq[0]:.4f}  {tq[1]:.4f}" + " " * 28 + f"{af[0]:.4f}  {af[1]:.4f}")

TurboQuant's distortion equals $d \cdot C(f_X, b)$ on both sets, since the rotation makes the input's shape irrelevant, while the affine baseline is several times worse on the outlier vectors at every bit-width and worse on the sphere below four bits. At four bits the two are close on the sphere; the rotation buys most at low bit-widths and on structured vectors.

## 4. Why a second step: bias

Minimizing $\lVert x - \tilde{x} \rVert^2$ does not make $\langle y, \tilde{x} \rangle$ an unbiased estimate of $\langle y, x \rangle$. The paper works out the one-bit case: $\mathbb{E}\,\langle y, \tilde{x} \rangle = \frac{2}{\pi}\langle y, x \rangle$, a shrinkage by $2/\pi \approx 0.64$. Attention scores are inner products that go through a softmax, so a systematic shrinkage flattens the attention distribution, and the paper notes the bias diminishes as the bit-width grows. The cell fits the slope of estimated against true inner products.

In [ ]:
Y_queries = unit(torch.randn(5000, d))
true_ip = (X_sphere * Y_queries).sum(dim=1)


def slope(estimate):
    return ((estimate * true_ip).sum() / (true_ip * true_ip).sum()).item()


for bits in range(1, 4):
    X_hat = tq_mse_dequantize(tq_mse_quantize(X_sphere, Pi, codebooks[bits]), Pi, codebooks[bits])
    note = f"   (paper: 2/pi = {2 / math.pi:.3f})" if bits == 1 else ""
    fitted = slope((X_hat * Y_queries).sum(dim=1))
    print(f"b={bits}: slope of <y, x_mse> against <y, x> = {fitted:.3f}{note}")

## 5. The residual correction

TurboQuant$_{\mathrm{prod}}$ spends $b - 1$ bits on the MSE quantizer and the last bit on the residual $r = x - \tilde{x}_{\mathrm{mse}}$, using a quantized Johnson-Lindenstrauss transform (Definition 1 and Algorithm 2 of the paper). With $S \in \mathbb{R}^{d \times d}$ of i.i.d. standard normal entries, shared by all vectors, the stored data are

$$\mathrm{idx} = \mathrm{Quant}_{\mathrm{mse}}(x), \qquad \mathrm{qjl} = \mathrm{sign}(S\,r) \in \{-1, +1\}^d, \qquad \gamma = \lVert r \rVert_2,$$

and the reconstruction is

$$\tilde{x} = \tilde{x}_{\mathrm{mse}} + \frac{\sqrt{\pi/2}}{d}\,\gamma\,S^T \mathrm{qjl}.$$

Lemma 4 states that the second term is an unbiased estimate of $r$ in every inner product, with variance at most $\frac{\pi}{2d}\lVert y \rVert^2 \lVert r \rVert^2$, and Theorem 2 that $\mathbb{E}\,\langle y, \tilde{x} \rangle = \langle y, x \rangle$ with distortion about $1.57/d$, $0.56/d$, $0.18/d$ and $0.047/d$ for $b = 1$ to $4$. The cell checks the slope, the inner-product distortion and the variance bound.

In [ ]:
S = torch.randn(d, d, generator=gen)


def tq_prod_quantize(X, Pi, levels, S):
    idx = tq_mse_quantize(X, Pi, levels) if levels is not None else None
    X_mse = tq_mse_dequantize(idx, Pi, levels) if levels is not None else torch.zeros_like(X)
    R = X - X_mse
    return idx, torch.sign(R @ S.T), R.norm(dim=1, keepdim=True)


def tq_prod_dequantize(idx, qjl, gamma, Pi, levels, S):
    X_mse = tq_mse_dequantize(idx, Pi, levels) if levels is not None else 0.0
    return X_mse + (math.sqrt(math.pi / 2) / d) * gamma * (qjl @ S)


print("b   slope   d * distortion of <y, x~>   paper   variance / bound")
for bits in range(1, 5):
    levels = codebooks[bits - 1] if bits > 1 else None     # b - 1 bits for the MSE stage
    idx, qjl, gamma = tq_prod_quantize(X_sphere, Pi, levels, S)
    X_hat = tq_prod_dequantize(idx, qjl, gamma, Pi, levels, S)
    est = (X_hat * Y_queries).sum(dim=1)
    # Variance of the residual estimate against Lemma 4's bound pi/(2d) * |y|^2 * |r|^2.
    X_mse = tq_mse_dequantize(idx, Pi, levels) if levels is not None else torch.zeros_like(X_sphere)
    resid_est = ((X_hat - X_mse) * Y_queries).sum(dim=1)
    resid_err = resid_est - ((X_sphere - X_mse) * Y_queries).sum(dim=1)
    var_ratio = (resid_err**2 / gamma.squeeze(1) ** 2).mean().item() / (math.pi / (2 * d))
    print(f"{bits}   {slope(est):.3f}   {d * (est - true_ip).pow(2).mean():24.3f}   "
          f"{[1.57, 0.56, 0.18, 0.047][bits - 1]:<6}  {var_ratio:.2f}")

The slope moves to within a few hundredths of 1 at every bit-width, where the MSE-only quantizer gave 0.63 to 0.97; the remaining gap is sampling noise of the estimator, whose variance is visible in the next column. The inner-product distortion matches the paper's values, and the variance of the residual term sits at the bound of Lemma 4, which is tight for Gaussian projections. The price of the correction is one of the $b$ bits and a norm per vector; the gain is that attention scores computed from the compressed keys are right on average.

## 6. What the paper reports

The paper evaluates KV-cache compression on LongBench with Llama-3.1-8B-Instruct and Ministral-7B-Instruct. Its Table 1 gives average scores of 50.06 for the full cache, 48.50 for KIVI at 3 bits, 50.16 for KIVI at 5 bits, 49.44 for TurboQuant at 2.5 bits and 50.06 for TurboQuant at 3.5 bits on Llama-3.1-8B-Instruct, and 49.89 against 49.62 at 2.5 bits on Ministral-7B-Instruct. Fractional bit-widths come from quantizing outlier channels at one more bit than the rest; the paper's example is 32 channels at 3 bits and 96 at 2 bits, an average of 2.5. On a needle-in-a-haystack test from 4k to 104k tokens it reports recall of 0.997 for both the full cache and TurboQuant. The text we read does not say whether keys, values or both were compressed in these runs, nor which of the two variants was used, so these numbers should be read as the paper's end-to-end result rather than as a measurement of either variant alone.

## Exercises

1. Replace the QR-based rotation by a randomized Hadamard transform (random signs followed by a Hadamard matrix), which costs $O(d \log d)$ instead of $O(d^2)$. Does the coordinate histogram of Section 2 change?
2. Fit the Lloyd-Max levels to $\mathcal{N}(0, 1/d)$ instead of $f_X$ and compare the distortion with Section 3 at $d = 16$ and $d = 128$. When does the difference matter?
3. Build 2000 random keys and one query, compute softmax attention weights from exact scores, from TurboQuant$_{\mathrm{mse}}$ at 2 bits and from TurboQuant$_{\mathrm{prod}}$ at 2 bits, and compare the three distributions. Which error does the softmax amplify?

## Summary

| Step | Operation | What it buys |
|------|-----------|--------------|
| Rotation | $\Pi x$ with $\Pi$ from QR of a Gaussian matrix | Every coordinate follows $f_X$; no per-vector scale |
| Scalar quantizer | nearest of $2^b$ Lloyd-Max levels for $f_X$ | Distortion within $\sqrt{3}\pi/2 \approx 2.7$ of the lower bound |
| Residual correction | one sign bit per coordinate, $S^T\mathrm{sign}(Sr)$ scaled by $\sqrt{\pi/2}\,\lVert r \rVert/d$ | Unbiased inner products, variance $\le \pi\lVert y\rVert^2\lVert r\rVert^2/(2d)$ |

Everything stored per vector is $b$ bits per coordinate plus one norm; the matrices $\Pi$ and $S$ and the levels are shared and fixed in advance, which is what lets the method quantize a token's key as soon as it is produced.